# Masking Inspector — New Collator (pr2)

Compare **legacy** (`instruction_template` + `response_template`) vs **span-based** (`assistant_turn` with `end_of_turn_template`) masking on the same tokenized inputs.

In [1]:
import json
import warnings

from transformers import AutoTokenizer

from oumi.core.collators.text_completions_collator_with_padding import (
    TextCompletionsCollatorWithPadding,
)

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print(f"Loaded tokenizer: {MODEL}")
print(
    f"Vocab size: {tokenizer.vocab_size}, pad={tokenizer.pad_token_id}, eos={tokenizer.eos_token_id}"
)

Loaded tokenizer: Qwen/Qwen2.5-1.5B-Instruct
Vocab size: 151643, pad=151643, eos=151645


In [2]:
def make_old_collator(tokenizer):
    """Legacy collator: instruction_template + response_template."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        return TextCompletionsCollatorWithPadding(
            tokenizer=tokenizer,
            instruction_template="<|im_start|>user\n",
            response_template="<|im_start|>assistant\n",
        )


def make_new_collator(tokenizer):
    """Span-based collator: assistant_turn with end_of_turn."""
    return TextCompletionsCollatorWithPadding(
        tokenizer=tokenizer,
        response_template="<|im_start|>assistant\n",
        end_of_turn_template="<|im_end|>",
        masking_method="assistant_turn",
    )


old_collator = make_old_collator(tokenizer)
new_collator = make_new_collator(tokenizer)
print("Old collator masking_method:", old_collator._default_collator.masking_method)
print("New collator masking_method:", new_collator._default_collator.masking_method)

Old collator masking_method: _legacy_instruction_response
New collator masking_method: assistant_turn


In [3]:
from IPython.display import HTML, display


def visualize_masking(input_ids, labels, tokenizer, title=""):
    """Render tokens color-coded: green=unmasked (trained on), gray=masked."""
    html = f"<h4>{title}</h4><div style='font-family:monospace; line-height:1.8; white-space:pre-wrap;'>"
    for tid, lab in zip(input_ids, labels):
        tok_str = (
            tokenizer.decode([tid])
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace(" ", "\u00b7")
        )
        if lab == -100:
            html += f"<span style='color:#999; background:#f0f0f0;'>{tok_str}</span>"
        else:
            html += f"<span style='color:#000; background:#b5f5b5; font-weight:bold;'>{tok_str}</span>"
    html += "</div>"
    display(HTML(html))


def compare_masking(messages, tokenizer, old_collator, new_collator, title=""):
    """Tokenize messages, run both collators, show side-by-side."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    input_ids = tokenizer.encode(text, add_special_tokens=False)
    batch = [{"input_ids": input_ids}]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        old_out = old_collator(batch)
        new_out = new_collator(batch)

    old_labels = old_out["labels"][0].tolist()
    new_labels = new_out["labels"][0].tolist()
    ids = old_out["input_ids"][0].tolist()

    old_unmasked = sum(1 for x in old_labels if x != -100)
    new_unmasked = sum(1 for x in new_labels if x != -100)
    diffs = sum(1 for a, b in zip(old_labels, new_labels) if (a == -100) != (b == -100))

    print(f"=== {title} ===")
    print(f"Total tokens: {len(ids)}")
    print(
        f"Old unmasked: {old_unmasked}, New unmasked: {new_unmasked}, Diff positions: {diffs}"
    )

    visualize_masking(ids, old_labels, tokenizer, title=f"{title} — OLD (legacy)")
    visualize_masking(ids, new_labels, tokenizer, title=f"{title} — NEW (span-based)")

    # Show specific diffs
    if diffs > 0:
        print(f"\nDiffering tokens ({diffs}):")
        for i, (a, b) in enumerate(zip(old_labels, new_labels)):
            if (a == -100) != (b == -100):
                tok = tokenizer.decode([ids[i]])
                old_s = "TRAINED" if a != -100 else "masked"
                new_s = "TRAINED" if b != -100 else "masked"
                print(f"  pos {i}: {repr(tok):20s}  old={old_s:8s}  new={new_s}")
    return old_labels, new_labels

## 1. Toy Data

In [4]:
# Single-turn
toy_single = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
]
compare_masking(toy_single, tokenizer, old_collator, new_collator, "Toy: single-turn");

=== Toy: single-turn ===
Total tokens: 34
Old unmasked: 8, New unmasked: 7, Diff positions: 1



Differing tokens (1):
  pos 33: '\n'                  old=TRAINED   new=masked


In [5]:
# Multi-turn
toy_multi = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
    {"role": "assistant", "content": "That would be 6."},
]
compare_masking(toy_multi, tokenizer, old_collator, new_collator, "Toy: multi-turn");

=== Toy: multi-turn ===
Total tokens: 56
Old unmasked: 16, New unmasked: 14, Diff positions: 2



Differing tokens (2):
  pos 33: '\n'                  old=TRAINED   new=masked
  pos 55: '\n'                  old=TRAINED   new=masked


In [6]:
# Tool-calling turn
toy_tool = [
    {"role": "system", "content": "You are a helpful assistant with tool access."},
    {"role": "user", "content": "What's the weather in SF?"},
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "get_weather", "arguments": {"city": "SF"}}</tool_call>',
    },
    {
        "role": "tool",
        "content": '<tool_response>{"temp": "65F", "condition": "sunny"}</tool_response>',
    },
    {"role": "assistant", "content": "It's 65F and sunny in San Francisco."},
]
compare_masking(toy_tool, tokenizer, old_collator, new_collator, "Toy: tool-calling");

=== Toy: tool-calling ===
Total tokens: 101
Old unmasked: 34, New unmasked: 32, Diff positions: 2



Differing tokens (2):
  pos 48: '\n'                  old=TRAINED   new=masked
  pos 100: '\n'                  old=TRAINED   new=masked


## 2. TatQA Data

In [7]:
TATQA_PATH = "/data/shanghong/oumi-pr2/tool_call_project/data/train_final_llama3.3_70b_instruct_max2048.jsonl"
with open(TATQA_PATH) as f:
    tatqa_samples = [json.loads(line) for line in f.readlines()[:5]]
print(f"Loaded {len(tatqa_samples)} TatQA samples")

for i, sample in enumerate(tatqa_samples[:3]):
    msgs = sample["messages"]
    preview = msgs[-1]["content"][:80]
    print(f"\nSample {i}: {len(msgs)} messages, last assistant: {preview}...")
    compare_masking(msgs, tokenizer, old_collator, new_collator, f"TatQA sample {i}")

Loaded 5 TatQA samples

Sample 0: 3 messages, last assistant: <think> 
To determine the years in which the financial information was recorded,...
=== TatQA sample 0 ===
Total tokens: 410
Old unmasked: 119, New unmasked: 118, Diff positions: 1



Differing tokens (1):
  pos 409: '\n'                  old=TRAINED   new=masked

Sample 1: 3 messages, last assistant: <think> To find the net debt receipts in 2019, we need to look at the Statement ...
=== TatQA sample 1 ===
Total tokens: 388
Old unmasked: 94, New unmasked: 93, Diff positions: 1



Differing tokens (1):
  pos 387: '\n'                  old=TRAINED   new=masked

Sample 2: 3 messages, last assistant: <think> To find the net debt repayments in 2018, we need to look at the Statemen...
=== TatQA sample 2 ===
Total tokens: 373
Old unmasked: 78, New unmasked: 77, Diff positions: 1



Differing tokens (1):
  pos 372: '\n'                  old=TRAINED   new=masked


## 3. Hermes Data

In [8]:
HERMES_PATH = "/data/shanghong/oumi-pr2/tool_call_project/data/hermes_reasoning_tool_use_train_split_train_filtered.jsonl"
with open(HERMES_PATH) as f:
    hermes_samples = [json.loads(line) for line in f.readlines()[:5]]
print(f"Loaded {len(hermes_samples)} Hermes samples")

for i, sample in enumerate(hermes_samples[:3]):
    msgs = sample["messages"]
    roles = [m["role"] for m in msgs]
    print(f"\nSample {i}: {len(msgs)} messages, roles: {roles}")
    compare_masking(msgs, tokenizer, old_collator, new_collator, f"Hermes sample {i}")

Loaded 5 Hermes samples

Sample 0: 9 messages, roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'user', 'assistant', 'tool', 'assistant']
=== Hermes sample 0 ===
Total tokens: 1509
Old unmasked: 409, New unmasked: 405, Diff positions: 4



Differing tokens (4):
  pos 884: '\n'                  old=TRAINED   new=masked
  pos 1133: '\n'                  old=TRAINED   new=masked
  pos 1268: '\n'                  old=TRAINED   new=masked
  pos 1508: '\n'                  old=TRAINED   new=masked

Sample 1: 8 messages, roles: ['system', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'assistant']
=== Hermes sample 1 ===
Total tokens: 1407
Old unmasked: 580, New unmasked: 573, Diff positions: 7



Differing tokens (7):
  pos 793: '\n'                  old=TRAINED   new=masked
  pos 1110: '\n'                  old=TRAINED   new=masked
  pos 1303: '\n'                  old=TRAINED   new=masked
  pos 1304: '<|im_start|>'        old=TRAINED   new=masked
  pos 1305: 'assistant'           old=TRAINED   new=masked
  pos 1306: '\n'                  old=TRAINED   new=masked
  pos 1406: '\n'                  old=TRAINED   new=masked

Sample 2: 3 messages, roles: ['system', 'user', 'assistant']
=== Hermes sample 2 ===
Total tokens: 1564
Old unmasked: 435, New unmasked: 434, Diff positions: 1



Differing tokens (1):
  pos 1563: '\n'                  old=TRAINED   new=masked


## Summary: Aggregate Diff Stats

In [9]:
# Run over more samples to get aggregate stats
def diff_stats(data_path, n=50):
    with open(data_path) as f:
        samples = [json.loads(line) for line in f.readlines()[:n]]
    total_diffs = 0
    total_tokens = 0
    total_old_unmasked = 0
    total_new_unmasked = 0
    for sample in samples:
        msgs = sample["messages"]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        input_ids = tokenizer.encode(text, add_special_tokens=False)
        batch = [{"input_ids": input_ids}]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            old_out = old_collator(batch)
            new_out = new_collator(batch)
        old_labels = old_out["labels"][0].tolist()
        new_labels = new_out["labels"][0].tolist()
        total_tokens += len(input_ids)
        total_old_unmasked += sum(1 for x in old_labels if x != -100)
        total_new_unmasked += sum(1 for x in new_labels if x != -100)
        total_diffs += sum(
            1 for a, b in zip(old_labels, new_labels) if (a == -100) != (b == -100)
        )
    print(f"  Samples: {len(samples)}")
    print(f"  Total tokens: {total_tokens}")
    print(f"  Old unmasked: {total_old_unmasked}")
    print(f"  New unmasked: {total_new_unmasked}")
    print(
        f"  Differing positions: {total_diffs} ({total_diffs / total_tokens * 100:.2f}%)"
    )


print("TatQA:")
diff_stats(TATQA_PATH)
print("\nHermes:")
diff_stats(HERMES_PATH)

TatQA:
  Samples: 50
  Total tokens: 37139
  Old unmasked: 8838
  New unmasked: 8788
  Differing positions: 50 (0.13%)

Hermes:
  Samples: 50
  Total tokens: 76201
  Old unmasked: 27408
  New unmasked: 27248
  Differing positions: 160 (0.21%)
